In [1]:
# Vectorstores + Embeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Document loading + splitting
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chat models (provider-specific)
from langchain_openai import ChatOpenAI   # or from langchain_groq import ChatGroq

# Prompts + Documents
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

# Chains (moved to langchain_classic in LangChain 1.x)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

C:\Users\Adit Sharma\AppData\Local\Temp\ipykernel_40048\1123372582.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("groq_api")


In [3]:
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [4]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
retriever=vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

In [9]:
from langchain.chat_models import init_chat_model
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")
llm=init_chat_model("groq:openai/gpt-oss-20b")

In [10]:
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [11]:
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 **LangChain supports agents and memory in two complementary ways:**

| Feature | What it does | How it’s implemented in LangChain |
|---------|--------------|-----------------------------------|
| **Agents** | LLM‑driven entities that decide *which* external tool to use and *in what order* to execute it while completing a task. | • **Tool‑based agents** – you register a set of tools (e.g., a calculator, a web‑search API, a database connector, or any custom function). <br>• The agent’s policy (e.g., `Sequential`, `ReAct`, `ZeroShot`) tells the LLM to “think” about the next step, call a tool, receive the result, and continue until the goal is satisfied. <br>• Agents can be chained or nested, so one agent can spawn another, enabling complex workflows. |
| **Memory** | The ability for the LLM to “remember” earlier turns of a conversation or prior results, making interactions coherent over multiple turns. | • **ConversationBufferMemory** – stores the raw dialogue history (questio

In [12]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='df8f307e-9c17-4ba4-925b-da555084fc8b', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='faa7b395-8f92-4bd1-a809-9ccba9bc27b7', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='c68876cd-fd7e-4f22-8612-d732d79e8c29', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order duri